# Quantum Maize Yield Model

This notebook predicts maize yield (t/ha) for nine Kenyan counties using only rainfall observed on or before 30 June of each year. The yield workbook has no 2019 observations in either sheet, so the chronological split is train = 2012-2017, validation = 2018, and test = 2020. The test set is intentionally small because it contains at most one row per county.

The quantum model is evaluated on its own terms. No claim about outperforming a classical baseline is made here.

In [ ]:
import importlib.util
import os
import sys
import subprocess

required = {
    'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn', 'qiskit': 'qiskit',
    'qiskit_machine_learning': 'qiskit-machine-learning',
    'qiskit_algorithms': 'qiskit-algorithms',
    'openpyxl': 'openpyxl', 'xlrd': 'xlrd',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Kernel:', sys.executable)
print('All required packages are available.')

Kernel: C:\Python314\python.exe
All required packages are available.


In [ ]:
import glob
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

DATA_DIR = Path('data') / 'Agr data'
YIELD_PATH = DATA_DIR / 'Annual Maize Yield Production 2012-2020.xlsx'
OUTPUT_DIR = Path('.')
NAME_MAP = {
    'BOMET': 'Bomet', 'BUNGOMA': 'Bungoma',
    'ELGEYO MARAKWET': 'Elgeyo/Marakwet', 'KAKAMEGA': 'Kakamega',
    'NAKURU': 'Nakuru', 'NANDI': 'Nandi', 'NAROK': 'Narok',
    'TRANSNZOIA': 'Trans Nzoia', 'TRANS NZOIA': 'Trans Nzoia',
    'UASIN GISHU': 'Uasin Gishu',
}
COUNTIES = list(dict.fromkeys(NAME_MAP.values()))
FEATURES = ['cum_rainfall_30jun', 'rainy_days_30jun', 'longest_dry_spell', 'rainfall_anomaly']
print('Working directory:', Path.cwd())
print('Data directory:', DATA_DIR.resolve())

Working directory: C:\Users\user\QS-Agric-T3
Data directory: C:\Users\user\QS-Agric-T3\data\Agr data


In [ ]:
def parse_sheet1(path):
    raw = pd.read_excel(path, sheet_name='Sheet1', header=None)
    year_row = raw.iloc[1]
    data = raw.iloc[3:].reset_index(drop=True)
    records = []
    n_cols = raw.shape[1]
    col = 1
    while col < n_cols:
        year = year_row[col]
        if pd.isna(year):
            col += 1
            continue
        harvested_col, production_col, yield_col = col, col + 1, col + 2
        for _, row in data.iterrows():
            county = row[0]
            if pd.isna(county):
                continue
            records.append({
                'county': str(county).strip(), 'year': int(year),
                'harvested_area_ha': row[harvested_col],
                'production_tonnes': row[production_col],
                'yield_t_ha': row[yield_col],
            })
        col += 3
    return pd.DataFrame(records)

def parse_sheet4(path):
    raw = pd.read_excel(path, sheet_name='Sheet4')
    raw.columns = ['county', 'year', 'indicator', 'value']
    pivoted = raw.pivot_table(index=['county', 'year'], columns='indicator',
                               values='value', aggfunc='first').reset_index()
    pivoted.columns.name = None
    pivoted = pivoted.rename(columns={
        'Area (HA)': 'harvested_area_ha',
        'Production (MT)': 'production_tonnes',
        'Yield(MT/HA)': 'yield_t_ha',
    })
    pivoted['county'] = pivoted['county'].str.strip()
    return pivoted

sheet1_df = parse_sheet1(YIELD_PATH)
sheet4_df = parse_sheet4(YIELD_PATH)
combined = pd.concat([sheet4_df, sheet1_df], ignore_index=True)
combined = combined.drop_duplicates(subset=['county', 'year'], keep='first')
combined = combined.sort_values(['county', 'year']).reset_index(drop=True)
combined = combined[combined['county'].isin(COUNTIES)].copy()
print('Sheet1 years:', sorted(sheet1_df['year'].unique()))
print('Sheet4 years:', sorted(sheet4_df['year'].unique()))
print('Filtered yield shape:', combined.shape)

Sheet1 years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020)]
Sheet4 years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020)]
Filtered yield shape: (72, 5)


In [ ]:
def load_rainfall(path):
    df = pd.read_excel(path, skiprows=1)
    df.columns = ['date', 'rainfall_mm']
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['rainfall_mm'] = pd.to_numeric(df['rainfall_mm'], errors='coerce')
    return df.dropna(subset=['date'])

def longest_dry_spell(values):
    longest = current = 0
    for value in values:
        if value < 1.0:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest

rainfall_features = []
rainfall_paths = glob.glob(str(DATA_DIR / '*MEAN DAILY RAINFALL*'))
for path_string in rainfall_paths:
    filename = Path(path_string).name.upper()
    matches = [key for key in NAME_MAP if filename.startswith(key + ' ') or filename.startswith(key + 'M')]
    if not matches:
        continue
    county = NAME_MAP[max(matches, key=len)]
    rain = load_rainfall(path_string)
    rain = rain[rain['date'].dt.year.isin(combined['year'].unique())].copy()
    for year, group in rain.groupby(rain['date'].dt.year):
        group = group[group['date'] <= pd.Timestamp(year=int(year), month=6, day=30)]
        if group.empty:
            continue
        rainfall_features.append({
            'county': county, 'year': int(year),
            'cum_rainfall_30jun': group['rainfall_mm'].sum(),
            'rainy_days_30jun': int((group['rainfall_mm'] >= 1.0).sum()),
            'longest_dry_spell': longest_dry_spell(group['rainfall_mm'].to_numpy()),
        })
rainfall_df = pd.DataFrame(rainfall_features).drop_duplicates(['county', 'year'])
dataset = combined.merge(rainfall_df, on=['county', 'year'], how='inner')
dataset = dataset.dropna(subset=['yield_t_ha'] + FEATURES[:3]).sort_values(['county', 'year']).reset_index(drop=True)
print('Final merged shape:', dataset.shape)
print('Year range:', int(dataset['year'].min()), 'to', int(dataset['year'].max()))
print('Years:', sorted(dataset['year'].unique()))
print('Counties:', sorted(dataset['county'].unique()))
print('Rows per county:')
print(dataset.groupby('county').size().to_string())

Final merged shape: (36, 8)
Year range: 2016 to 2020
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020)]
Counties: ['Bomet', 'Bungoma', 'Elgeyo/Marakwet', 'Kakamega', 'Nakuru', 'Nandi', 'Narok', 'Trans Nzoia', 'Uasin Gishu']
Rows per county:
county
Bomet              4
Bungoma            4
Elgeyo/Marakwet    4
Kakamega           4
Nakuru             4
Nandi              4
Narok              4
Trans Nzoia        4
Uasin Gishu        4


In [ ]:
train = dataset[dataset['year'] <= 2017].copy()
validation = dataset[dataset['year'] == 2018].copy()
test = dataset[dataset['year'] == 2020].copy()
train_rainfall_means = train.groupby('county')['cum_rainfall_30jun'].mean()
for frame in (train, validation, test):
    frame['rainfall_anomaly'] = frame.apply(
        lambda row: row['cum_rainfall_30jun'] - train_rainfall_means.loc[row['county']], axis=1
    )

scaler = StandardScaler()
X_train = scaler.fit_transform(train[FEATURES])
X_validation = scaler.transform(validation[FEATURES])
X_test = scaler.transform(test[FEATURES])
y_train = train['yield_t_ha'].to_numpy(dtype=float)
y_validation = validation['yield_t_ha'].to_numpy(dtype=float)
y_test = test['yield_t_ha'].to_numpy(dtype=float)
print(f'Train rows: {len(train)}, validation rows: {len(validation)}, test rows: {len(test)}')
print('Small test set note: one 2020 observation per county is possible, so county-level test results are based on very few observations.')

feature_map = ZZFeatureMap(feature_dimension=len(FEATURES), reps=2, entanglement='full')
kernel = FidelityQuantumKernel(feature_map=feature_map)
K_train = kernel.evaluate(x_vec=X_train)
K_validation = kernel.evaluate(x_vec=X_validation, y_vec=X_train)
lambda_grid = [0.01, 0.1, 1.0, 10.0]
validation_rows = []
for lam in lambda_grid:
    alpha = np.linalg.solve(K_train + lam * np.eye(len(X_train)), y_train)
    validation_rows.append({'lambda': lam, 'validation_MAE': mean_absolute_error(y_validation, K_validation @ alpha)})
lambda_results = pd.DataFrame(validation_rows)
best_lambda = float(lambda_results.loc[lambda_results['validation_MAE'].idxmin(), 'lambda'])
print('Validation lambda search:')
print(lambda_results.to_string(index=False))
print('Selected lambda:', best_lambda)

Train rows: 18, validation rows: 9, test rows: 9
Small test set note: one 2020 observation per county is possible, so county-level test results are based on very few observations.


C:\Users\user\AppData\Local\Temp\ipykernel_17156\3560393568.py:20: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=len(FEATURES), reps=2, entanglement='full')


Validation lambda search:
 lambda  validation_MAE
   0.01        1.633731
   0.10        1.703051
   1.00        2.141592
  10.00        2.894409
Selected lambda: 0.01


In [ ]:
def metric_row(label, actual, predicted):
    return {
        'group': label,
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted) if len(actual) > 1 else np.nan,
        'n': len(actual),
    }

start_time = time.perf_counter()
alpha = np.linalg.solve(K_train + best_lambda * np.eye(len(X_train)), y_train)
K_test = kernel.evaluate(x_vec=X_test, y_vec=X_train)
test_predictions = K_test @ alpha
test_df = test.copy()
test_df['pred_quantum'] = test_predictions
test_df['year'] = pd.to_numeric(test_df['year'], errors='raise').astype('int64')
predictions = test_df[['county', 'year', 'yield_t_ha', 'pred_quantum']].copy()
predictions = predictions.rename(columns={'county': 'County', 'year': 'Year', 'yield_t_ha': 'Actual', 'pred_quantum': 'Quantum_prediction'})
metric_rows = [metric_row('Overall', predictions['Actual'], predictions['Quantum_prediction'])]
for county in ['Bungoma', 'Nandi'] + [c for c in COUNTIES if c not in ['Bungoma', 'Nandi']]:
    subset = predictions[predictions['County'] == county]
    if not subset.empty:
        metric_rows.append(metric_row(county, subset['Actual'], subset['Quantum_prediction']))
metrics = pd.DataFrame(metric_rows)
print('Final test metrics (quantum model):')
print(metrics.to_string(index=False, float_format=lambda value: f'{value:.6f}'))

import matplotlib.pyplot as plt

def plot_actual_vs_predicted(test_df, county, pred_col, pred_label, filename):
    sub = test_df[test_df['county'] == county].copy()
    sub['year'] = pd.to_numeric(sub['year'], errors='raise').astype('int64')
    x_min = int(sub['year'].min()) - 1
    x_max = int(sub['year'].max()) + 1

    plt.figure(figsize=(7, 4))
    plt.scatter(sub['year'], sub['yield_t_ha'], color='tab:blue', s=80,
                label='Actual', zorder=3)
    plt.scatter(sub['year'], sub[pred_col], color='tab:orange', marker='s',
                s=80, label=pred_label, zorder=3)
    plt.xlim(x_min, x_max)
    plt.xticks(sub['year'].unique())
    plt.title(f'{county}: actual vs predicted maize yield')
    plt.xlabel('Year')
    plt.ylabel('Yield (t/ha)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    print(f'{county} plot x-limits: {x_min}, {x_max}; year dtype: {sub["year"].dtype}')
    plt.show()

for county in ['Bungoma', 'Nandi']:
    plot_actual_vs_predicted(test_df, county, 'pred_quantum',
                             'Quantum prediction',
                             f'actual_vs_predicted_{county}.png')

predictions.to_csv(OUTPUT_DIR / 'quantum_predictions.csv', index=False)
print('Saved predictions:', OUTPUT_DIR / 'quantum_predictions.csv')
print('Predictions CSV first 10 rows:')
print(predictions.head(10).to_string(index=False))
closed_form_runtime = time.perf_counter() - start_time

Final test metrics (quantum model):
          group      MAE     RMSE        R2  n
        Overall 1.389958 1.553316 -1.486452  9
        Bungoma 2.034604 2.034604       NaN  1
          Nandi 1.706821 1.706821       NaN  1
          Bomet 0.307482 0.307482       NaN  1
Elgeyo/Marakwet 2.148808 2.148808       NaN  1
       Kakamega 1.538192 1.538192       NaN  1
         Nakuru 0.296623 0.296623       NaN  1
          Narok 1.224050 1.224050       NaN  1
    Trans Nzoia 1.032018 1.032018       NaN  1
    Uasin Gishu 2.221026 2.221026       NaN  1


Saved plot: bungoma_actual_vs_predicted.png


Saved plot: nandi_actual_vs_predicted.png
Saved predictions: quantum_predictions.csv
Predictions CSV first 10 rows:
         County  Year   Actual  Quantum_prediction
          Bomet  2020 1.928822            1.621341
        Bungoma  2020 3.614283            1.579679
Elgeyo/Marakwet  2020 3.381469            1.232661
       Kakamega  2020 2.708367            1.170175
         Nakuru  2020 1.421650            1.125027
          Nandi  2020 3.173353            1.466532
          Narok  2020 2.816155            1.592105
    Trans Nzoia  2020 0.605191            1.637209
    Uasin Gishu  2020 3.545068            1.324042


In [7]:
seed_values = [7, 17, 27, 37, 47]
robustness_rows = []
for seed in seed_values:
    seed_start = time.perf_counter()
    sampler = StatevectorSampler(default_shots=1024, seed=seed)
    fidelity = ComputeUncompute(sampler=sampler)
    shot_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    shot_K_train = shot_kernel.evaluate(x_vec=X_train)
    shot_K_test = shot_kernel.evaluate(x_vec=X_test, y_vec=X_train)
    shot_alpha = np.linalg.solve(shot_K_train + best_lambda * np.eye(len(X_train)), y_train)
    shot_pred = shot_K_test @ shot_alpha
    robustness_rows.append({'seed': seed, 'MAE': mean_absolute_error(y_test, shot_pred), 'runtime_seconds': time.perf_counter() - seed_start})
robustness = pd.DataFrame(robustness_rows)
print('5-seed shot-noise robustness results (1024 shots per kernel evaluation):')
print(robustness.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('MAE mean +/- std: {:.6f} +/- {:.6f}'.format(robustness['MAE'].mean(), robustness['MAE'].std(ddof=1)))
resource_table = pd.DataFrame([{'qubits': len(FEATURES), 'encoding': 'ZZFeatureMap', 'reps': feature_map.reps, 'circuit_depth': feature_map.decompose().depth(), 'shots': 1024, 'optimiser': 'none (closed-form ridge)', 'parameter_count': feature_map.num_parameters, 'seeds': ', '.join(map(str, seed_values)), 'runtime_seconds': closed_form_runtime + robustness['runtime_seconds'].sum(), 'backend': 'statevector simulator'}])
print('Quantum resource table:')
print(resource_table.to_string(index=False))

5-seed shot-noise robustness results (1024 shots per kernel evaluation):
 seed      MAE  runtime_seconds
    7 1.384586         8.627520
   17 1.405467         7.968316
   27 1.374274         7.995157
   37 1.418223         9.306861
   47 1.414961         8.727818
MAE mean +/- std: 1.399502 +/- 0.019261
Quantum resource table:
 qubits     encoding  reps  circuit_depth  shots                optimiser  parameter_count             seeds  runtime_seconds               backend
      4 ZZFeatureMap     2             31   1024 none (closed-form ridge)                4 7, 17, 27, 37, 47        47.256786 statevector simulator


## Claim discipline

This notebook reports only the quantum model's observed MAE, RMSE, R2, seed robustness, and runtime.

**Comparison with classical baseline (MAE = ___, pending Samuel Muoria's notebook)**

In [ ]:
# 2021 forecast: emit a prediction only when rainfall through 30 June is provided.
forecast_year = 2021
forecast_rows = []
for county in ['Bungoma', 'Nandi']:
    county_paths = []
    for path_string in rainfall_paths:
        filename = Path(path_string).name.upper()
        matches = [key for key in NAME_MAP if filename.startswith(key + ' ') or filename.startswith(key + 'M')]
        if matches and NAME_MAP[max(matches, key=len)] == county:
            county_paths.append(path_string)
    if not county_paths:
        forecast_rows.append({'County': county, 'Year': forecast_year, 'Quantum_prediction': np.nan, 'Forecast_status': 'No rainfall file'})
        continue
    rain = load_rainfall(county_paths[0])
    group = rain[(rain['date'].dt.year == forecast_year) & (rain['date'] <= pd.Timestamp(year=forecast_year, month=6, day=30))].copy()
    if group.empty:
        forecast_rows.append({'County': county, 'Year': forecast_year, 'Quantum_prediction': np.nan, 'Forecast_status': 'No rainfall through 30 June'})
        continue
    row = {
        'County': county, 'Year': forecast_year,
        'cum_rainfall_30jun': group['rainfall_mm'].sum(),
        'rainy_days_30jun': int((group['rainfall_mm'] >= 1.0).sum()),
        'longest_dry_spell': longest_dry_spell(group['rainfall_mm'].to_numpy()),
        'Forecast_status': 'Prediction available',
    }
    row['rainfall_anomaly'] = row['cum_rainfall_30jun'] - train_rainfall_means.loc[county]
    forecast_rows.append(row)

forecast_features = pd.DataFrame(forecast_rows)
forecast_predictions = forecast_features[['County', 'Year', 'Forecast_status']].copy()
available = forecast_features['Forecast_status'].eq('Prediction available')
forecast_predictions['Quantum_prediction'] = np.nan
if available.any():
    X_future = scaler.transform(forecast_features.loc[available, FEATURES])
    K_future = kernel.evaluate(x_vec=X_future, y_vec=X_train)
    forecast_predictions.loc[available, 'Quantum_prediction'] = K_future @ alpha
forecast_predictions = forecast_predictions.sort_values(['County', 'Year']).reset_index(drop=True)
forecast_predictions.to_csv('quantum_forecasts_2021.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
for county, colour in [('Bungoma', '#1f77b4'), ('Nandi', '#d62728')]:
    subset = forecast_predictions[forecast_predictions['County'] == county]
    available_subset = subset.dropna(subset=['Quantum_prediction'])
    if available_subset.empty:
        ax.plot([], [], marker='o', color=colour, label=f'{county} (unavailable)')
        ax.text(forecast_year, 0.05, f'{county}: no 2021 rainfall data', color=colour,
                ha='center', transform=ax.get_xaxis_transform())
    else:
        ax.scatter(available_subset['Year'].astype(int), available_subset['Quantum_prediction'],
                   s=90, color=colour, label=county, zorder=3)
ax.set_xlim(forecast_year - 1, forecast_year + 1)
ax.set_xticks([forecast_year])
ax.set(title='Quantum maize-yield forecast for 2021', xlabel='Year', ylabel='Predicted yield (t/ha)')
ax.grid(alpha=0.3)
ax.legend(title='County')
fig.tight_layout()
fig.savefig('bungoma_nandi_forecast_2021.png', dpi=180)
plt.show()
print('2021 forecast coverage and predictions:')
print(forecast_predictions.to_string(index=False))
print('Saved:', 'quantum_forecasts_2021.csv', 'and', 'bungoma_nandi_forecast_2021.png')

In [ ]:
# Fresh plotting verification cell: integer-year scatter plots for the one-year test set.
import matplotlib.pyplot as plt

plot_test_df = test_df.copy()
plot_test_df['year'] = pd.to_numeric(plot_test_df['year'], errors='raise').astype(int)
for county in ['Bungoma', 'Nandi']:
    sub = plot_test_df[plot_test_df['county'] == county].copy()
    plt.figure(figsize=(7, 4))
    plt.scatter(sub['year'], sub['yield_t_ha'], color='tab:blue', s=80,
                label='Actual', zorder=3)
    plt.scatter(sub['year'], sub['pred_quantum'], color='tab:orange', marker='s',
                s=80, label='Quantum prediction', zorder=3)
    x_min = int(sub['year'].min()) - 1
    x_max = int(sub['year'].max()) + 1
    plt.xlim(x_min, x_max)
    plt.xticks(sub['year'].unique())
    plt.title(f'{county}: actual vs predicted maize yield')
    plt.xlabel('Year')
    plt.ylabel('Yield (t/ha)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    filename = f'actual_vs_predicted_{county}.png'
    plt.savefig(filename, dpi=150)
    print(f'Saved {filename}; x-limits=({x_min}, {x_max}); year dtype={sub["year"].dtype}')
    plt.show()